# Rain, infiltration, ponding, exfiltration and evaporation with the VSSB surface boundary

![VSSB surface processes: precipitation, infiltration, ponding, exfiltration, evaporation and runoff on a soil column](surface_boundary_processes.png)

A deliberately simple, transient experiment with the development **VSSB** (variably saturated surface boundary) package of MODFLOW 6. A single homogeneous soil column is subjected to a sequence of weather and groundwater conditions, so that we can focus on how the surface processes interact:

| Process | In VSSB |
|---|---|
| precipitation | prescribed rate $P$ on the top face of the surface cell |
| infiltration | exchange with the soil through the top face, limited by the soil: $Q_\mathrm{inf}=C_s\,(H_s-h)$ with $C_s=k_r K A/(\Delta z/2)$ |
| ponding | water that cannot infiltrate is stored in a pond of depth $d\le d_\mathrm{max}$; the pond head $H_s=z_s+d$ is the driving head |
| exfiltration | when $h>H_s$ the same exchange becomes negative: groundwater flows into the pond |
| runoff | water that cannot be stored (pond full) leaves the package |
| evaporation | a potential rate $E$ is removed first from the pond at the potential rate, then the remaining demand is taken from the soil as bare-soil evaporation, reduced by the relative permeability $k_r$ of the top cell |

None of infiltration, ponding, runoff, exfiltration or the split of the evaporation is prescribed: they follow from the Richards solution in the top cell, the rain, the potential evaporation and the pond. The potential evaporation $E$ is a demand that is met in two steps and never exceeded, so the total evaporation $E_\mathrm{pond}+E_\mathrm{soil}\le E$. The pond evaporation is part of the pond water balance, while the bare-soil evaporation is extracted directly from the top soil cell and is folded into the surface exchange with the aquifer. Per time step the pond budget closes exactly,

$$P = Q_\mathrm{inf} + Q_\mathrm{runoff} + \frac{\Delta V_\mathrm{pond}}{\Delta t} + E_\mathrm{pond}.$$

(Transpiration/root-water uptake is a separate process, handled by the VSET package; VSSB simulates only evaporation.)

**Requirements:** this repository's Pixi kernel, a development `mf6` executable that includes VSSB with evaporation (`$MODFLOW_ROOT`, branch `vsf-dev`) and FloPy classes generated from the matching definition files (`pixi run update-flopy`), which provide `ModflowGwfvsf` and `ModflowGwfvssb` with a `precipitation` **and** `evaporation` entry in the period data. Model files go to `.output/surface_boundary/` under `examples/`.

In [ ]:
import shutil
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for package in ("ModflowGwfvsf", "ModflowGwfvssb"):
    if not hasattr(flopy.mf6, package):
        raise RuntimeError(
            f"FloPy class {package} missing; generate the classes from the Richards branch (pixi run update-flopy)."
        )

mf6_exe = shutil.which("mf6")
if mf6_exe is None:
    raise FileNotFoundError(
        "Richards mf6 executable not found on PATH; run in the Pixi environment."
    )

print(f"FloPy {flopy.__version__}; executable: {Path(mf6_exe).resolve()}")

## The experiment

Units are **m** and **day**; elevation $z$ is positive upward and the land surface is $z=0$. The column is 0.5 m deep, 1 m$^2$ in area, with 1 cm cells, no-flow sides and a **constant head at the bottom cell** that represents the regional groundwater level. That head is the only groundwater forcing; it is lowered and raised between phases. The initial state is hydrostatic with the water table 0.3 m below the surface.

The soil is illustrative (not calibrated): a sandy van Genuchten soil with $K_\mathrm{sat}=0.3$ m/day. Rain rates are chosen relative to it: 0.1 m/day (below), 1 m/day (above) and 0.3 m/day (equal) $K_\mathrm{sat}$. Evaporation rates are chosen the same way: a high 0.3 m/day to drain the pond and a more moderate 0.1 m/day to dry the bare soil. The intensities are high, to keep the experiment short. The seven phases are:

1. **Light rain**: everything infiltrates, no pond.
2. **Storm**, far more than the soil can take: the pond fills to $d_\mathrm{max}$ within hours and the rest runs off (infiltration excess).
3. **Pond evaporation** without rain: the full pond drains by *both* infiltration and evaporation. While water is ponded the evaporation runs at the full potential rate; once the pond is gone the demand is passed to the soil.
4. **Soil drying** without rain or pond: the potential evaporation is applied to the bare soil, where it is reduced by the relative permeability $k_r$ of the top cell and becomes supply-limited as the surface dries.
5. **Groundwater rise** without rain: the bottom head is raised above the land surface. The water table comes up to the surface and groundwater *exfiltrates* into the pond. Exfiltration slows down as the pond deepens ($H_s=z_s+d$ rises towards the soil head), and once the pond is full the seepage leaves as runoff.
6. **Rain on a seeping surface**: the pond is already full, so all rain becomes runoff (saturation excess) while exfiltration continues.
7. **Groundwater recedes**: the bottom head is lowered again, the seepage stops and the pond infiltrates.

Edit the table in the next cell to try other sequences. All rates are in m/day and the groundwater head is the constant head [m] in the bottom cell.

In [ ]:
column_length = 0.5  # m
nlay = 50
area = 1.0  # m2

ksat = 0.3  # m/day
porosity = 0.4
residual_saturation = 0.1  # saturation, NOT residual water content
alpha_vg = 5.0  # 1/m
n_vg = 2.5

max_pond_depth = 0.05  # m
initial_groundwater_head = -0.3  # m, hydrostatic start
steps_per_day = 100

schedule = pd.DataFrame(
    [
        # phase, duration [day], precipitation [m/day], potential evaporation [m/day], groundwater head at the bottom [m]
        ("1 light rain", 1.0, 0.10, 0.0, -0.3),
        ("2 storm", 0.5, 1.00, 0.0, -0.3),
        ("3 pond evaporation", 1.5, 0.0, 0.30, -0.3),
        ("4 soil drying", 2.0, 0.0, 0.10, -0.3),
        ("5 groundwater rise", 5.0, 0.0, 0.0, 0.15),
        ("6 rain on seepage", 0.5, 0.30, 0.0, 0.15),
        ("7 groundwater recedes", 3.0, 0.0, 0.0, -0.3),
    ],
    columns=["phase", "duration", "precipitation", "evaporation", "groundwater_head"],
)

if (
    nlay < 3
    or column_length <= 0
    or area <= 0
    or steps_per_day < 1
    or max_pond_depth < 0
):
    raise ValueError(
        "Use at least 3 layers, positive geometry, steps_per_day >= 1 and a nonnegative pond depth."
    )
if not (
    ksat > 0
    and 0 < porosity < 1
    and 0 <= residual_saturation < 1
    and alpha_vg > 0
    and n_vg > 1
):
    raise ValueError("Invalid conductivity, porosity or van Genuchten parameters.")
if (
    (schedule["duration"] <= 0).any()
    or (schedule["precipitation"] < 0).any()
    or (schedule["evaporation"] < 0).any()
):
    raise ValueError(
        "Durations must be positive and precipitation and evaporation nonnegative."
    )

cwd = Path.cwd().resolve()
examples_dir = None
for directory in (cwd, *cwd.parents):
    candidate = directory if directory.name == "examples" else directory / "examples"
    if (candidate / "notebooks" / "vsf-1-surface-boundary.ipynb").is_file():
        examples_dir = candidate
        break
if examples_dir is None:
    raise FileNotFoundError(
        f"Could not locate examples/notebooks/vsf-1-surface-boundary.ipynb from {cwd}."
    )
workspace = Path("models/vsf-surface-bnd")

dz = column_length / nlay
botm = -np.arange(1, nlay + 1) * dz
z = botm + dz / 2
nsteps_phase = np.rint(schedule["duration"] * steps_per_day).astype(int).to_numpy()
phase_end = schedule["duration"].cumsum().to_numpy()
phase_start = phase_end - schedule["duration"].to_numpy()
cs_sat = ksat * area / (dz / 2)  # surface conductance at kr = 1 [m2/day]

print(
    f"{nlay} cells; dz = {dz * 100:g} cm; {nsteps_phase.sum()} time steps over {phase_end[-1]:g} days"
)
print(f"Saturated surface conductance Cs = {cs_sat:g} m2/day (= Ksat*A/(dz/2))")
schedule

## The soil functions

The van Genuchten–Mualem relations below are exactly what the **VSF** package evaluates in every cell (`soil_model="vangenuchten"`): the **water-retention curve** maps pressure head $\psi$ to saturation $S$ (and water content $\theta=\phi\,S$), the **relative permeability** $k_r$ scales $K_\mathrm{sat}$ for unsaturated flow, and the **specific moisture capacity** $C=\mathrm{d}\theta/\mathrm{d}\psi$ is the storage term that makes Richards' equation nonlinear. With $m=1-1/n$ and effective saturation $S_e=\bigl[1+(-\alpha\psi)^n\bigr]^{-m}$ (and $S_e=1$ for $\psi\ge0$):

$$S = S_r + (1-S_r)\,S_e,\qquad k_r = \sqrt{S_e}\,\bigl[1-(1-S_e^{1/m})^m\bigr]^2,\qquad C = \phi\,(1-S_r)\,\frac{m\,n\,\alpha\,(-\alpha\psi)^{\,n-1}}{\bigl[1+(-\alpha\psi)^n\bigr]^{m+1}}.$$

These curves use the soil parameters set above; they shape every surface process in this notebook (how fast water infiltrates, how strongly the pond or the bare soil can be dried, and how sharp the wetting front is).

In [ ]:
# van Genuchten-Mualem soil functions, exactly as VSF evaluates them (VsfVanGenuchten.f90)
m_vg = 1.0 - 1.0 / n_vg


def effective_saturation(psi):
    psi = np.asarray(psi, dtype=float)
    return np.where(psi < 0.0, (1.0 + (-alpha_vg * psi) ** n_vg) ** (-m_vg), 1.0)


def saturation(psi):
    return residual_saturation + (1.0 - residual_saturation) * effective_saturation(psi)


def relative_permeability(psi):
    se = effective_saturation(psi)
    return np.sqrt(se) * (1.0 - (1.0 - se ** (1.0 / m_vg)) ** m_vg) ** 2


def moisture_capacity(psi):
    psi = np.asarray(psi, dtype=float)
    num = m_vg * n_vg * alpha_vg * (-alpha_vg * psi) ** (n_vg - 1.0)
    denom = (1.0 + (-alpha_vg * psi) ** n_vg) ** (m_vg + 1.0)
    return np.where(
        psi < 0.0, porosity * (1.0 - residual_saturation) * num / denom, 0.0
    )


suction = np.logspace(-3, 1, 400)  # -psi [m], wet (left) to dry (right)
psi = -suction

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

ax = axes[0]
ax.semilogx(suction, saturation(psi), color="tab:blue", lw=2)
ax.axhline(residual_saturation, color="0.5", ls=":", lw=1)
ax.text(suction[-1], residual_saturation + 0.02, "$S_r$", ha="right", color="0.4")
ax.set_xlabel("suction $-\\psi$ [m]")
ax.set_ylabel("saturation $S$ [-]")
ax.set_title("Water retention")
sec = ax.secondary_yaxis(
    "right", functions=(lambda s: s * porosity, lambda t: t / porosity)
)
sec.set_ylabel("water content $\\theta$ [-]")

ax = axes[1]
ax.loglog(suction, ksat * relative_permeability(psi), color="tab:green", lw=2)
ax.axhline(ksat, color="0.5", ls=":", lw=1)
ax.text(suction[0], ksat * 1.1, "$K_\\mathrm{sat}$", color="0.4")
ax.set_xlabel("suction $-\\psi$ [m]")
ax.set_ylabel("conductivity $K = K_\\mathrm{sat}\\,k_r$ [m/day]")
ax.set_title("Unsaturated conductivity")
ax.set_ylim(ksat * 1e-8, ksat * 2)

ax = axes[2]
ax.semilogx(suction, moisture_capacity(psi), color="tab:red", lw=2)
ax.set_xlabel("suction $-\\psi$ [m]")
ax.set_ylabel("moisture capacity $C = \\mathrm{d}\\theta/\\mathrm{d}\\psi$ [1/m]")
ax.set_title("Specific moisture capacity")

for ax in axes:
    ax.grid(which="both", alpha=0.25)
fig.suptitle(
    f"van Genuchten soil: $K_\\mathrm{{sat}}$={ksat} m/day, $\\phi$={porosity}, "
    f"$S_r$={residual_saturation}, $\\alpha$={alpha_vg} 1/m, $n$={n_vg}  ($m$={m_vg:g})",
    fontsize=11,
)
plt.show()

## Build and run

The soil is a **VSF** Richards soil with a uniform van Genuchten parameter set; NPF has fixed geometry (`icelltype=0`) and zero elastic storage, so changes in stored water are changes in saturation. The Newton formulation is used because VSSB's smoothed `min()` branches are Newton-linearized.

**VSSB** needs one surface cell (`packagedata`: id, cell, `max_pond_depth`) and, per stress period (`perioddata`), a precipitation rate **and** a potential evaporation rate per surface cell. Each phase is one stress period, which sets the precipitation, the potential evaporation and the bottom constant head. The package observations give per-step rates of precipitation, infiltration, runoff, pond storage and total evaporation, plus the pond depth. The cell-by-cell budget additionally stores the net surface exchange with the aquifer (the `VSSB` record), which is needed to separate the bare-soil evaporation from the infiltration.

In [ ]:
model_name = "column"
obs_types = [
    "precipitation",
    "infiltration",
    "runoff",
    "pond-storage",
    "evaporation",
    "pond-depth",
]

sim = flopy.mf6.MFSimulation(
    sim_name="surface_boundary", exe_name=mf6_exe, sim_ws=workspace
)
flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=len(schedule),
    perioddata=[(d, n, 1.0) for d, n in zip(schedule["duration"], nsteps_phase)],
)
gwf = flopy.mf6.ModflowGwf(
    sim, modelname=model_name, save_flows=True, newtonoptions="newton"
)
ims = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    complexity="COMPLEX",
    outer_maximum=300,
    inner_maximum=300,
    outer_dvclose=1e-8,
    inner_dvclose=1e-9,
)
sim.register_ims_package(ims, [gwf.name])
flopy.mf6.ModflowGwfdis(
    gwf,
    length_units="METERS",
    nlay=nlay,
    nrow=1,
    ncol=1,
    delr=np.sqrt(area),
    delc=np.sqrt(area),
    top=0.0,
    botm=botm,
)
flopy.mf6.ModflowGwfic(gwf, strt=initial_groundwater_head)
flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=ksat, k33=ksat)
flopy.mf6.ModflowGwfsto(gwf, ss=0.0, iconvert=0, transient={0: True})
flopy.mf6.ModflowGwfvsf(
    gwf,
    ivarsat=1,
    soil_model="vangenuchten",
    storage_scheme="chord-slope",
    kr_averaging="upstream",
    porosity=porosity,
    satres=residual_saturation,
    alphavgn=alpha_vg,
    nvgn=n_vg,
    sat_filerecord=f"{model_name}.sat",
    phead_filerecord=f"{model_name}.phd",
)
flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data={
        i: [[(nlay - 1, 0, 0), head]]
        for i, head in enumerate(schedule["groundwater_head"])
    },
    save_flows=True,
)
flopy.mf6.ModflowGwfvssb(
    gwf,
    nsurfcells=1,
    packagedata=[
        [0, (0, 0, 0), max_pond_depth]
    ],  # surface cell 0 (FloPy counts from zero)
    perioddata={
        i: [[0, precip, evap]]
        for i, (precip, evap) in enumerate(
            zip(schedule["precipitation"], schedule["evaporation"])
        )
    },
    print_flows=True,
    save_flows=True,
    observations={
        f"{model_name}.vssb.obs.csv": [(name, name, 1) for name in obs_types]
    },
)
flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{model_name}.hds",
    budget_filerecord=f"{model_name}.cbc",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)
sim.write_simulation(silent=True)
success, run_log = sim.run_simulation(silent=True, report=True)
if not success:
    raise RuntimeError(f"MODFLOW failed in {workspace}\n" + "\n".join(run_log[-40:]))
print(f"Simulation completed normally: {workspace}")

## Read the results and check the water balance

VSSB observations are values at the **end of each time step**, with the package's sign convention: `infiltration` is positive into the soil (negative = exfiltration), while `runoff`, `pond-storage` and `evaporation` are negative when they take water *out of the package*. Their rates are volumes per day; dividing by the cell area gives m/day. We flip the signs of runoff, pond storage and evaporation so that all plotted quantities are positive when they are what one expects: runoff leaving, pond filling, water evaporating.

The `evaporation` observation is the **total** evaporation (pond plus soil). To split it we use the `VSSB` cell-by-cell record, which is the net **surface exchange** with the aquifer: infiltration minus the bare-soil evaporation that is pulled from the top cell. The bare-soil part is therefore the difference between the `infiltration` observation and that surface exchange, and the pond part is the rest of the total.

Five independent checks are made:

1. the pond budget $P = Q_\mathrm{inf}+Q_\mathrm{runoff}+\Delta V_\mathrm{pond}/\Delta t+E_\mathrm{pond}$ closes in each step, and the split never produces a negative pond or soil evaporation, nor a total above the potential rate,
2. the pond depth equals the time integral of the pond storage and never exceeds $d_\mathrm{max}$; runoff only occurs with a full pond,
3. the change in soil water (porosity $\times$ saturation, from the VSF saturation file) equals the net surface exchange (infiltration minus bare-soil evaporation) plus the flow through the bottom constant head, read from the cell-budget file,
4. at the end of the storm and of the groundwater rise the column is saturated; if the pond is then full, the exchange must equal Darcy's law between the pond surface and the bottom cell, $Q_\mathrm{inf}/A=K_\mathrm{sat}(d_\mathrm{max}-h_\mathrm{bottom})/L_\mathrm{eff}$ with $L_\mathrm{eff}$ the distance from the surface to the bottom cell center. This is negative (exfiltration) when the groundwater head exceeds the pond surface.

In [ ]:
obs = pd.read_csv(workspace / f"{model_name}.vssb.obs.csv").set_index("time")
times = obs.index.to_numpy()
dt = np.diff(np.r_[0.0, times])

with flopy.utils.HeadFile(
    workspace / f"{model_name}.hds", precision="double"
) as reader:
    np.testing.assert_allclose(reader.get_times(), times)
    head = reader.get_alldata()[:, :, 0, 0]
with flopy.utils.HeadFile(
    workspace / f"{model_name}.phd", text="PRESSHEAD", precision="double"
) as reader:
    pressure_head = reader.get_alldata()[:, :, 0, 0]
with flopy.utils.HeadFile(
    workspace / f"{model_name}.sat", text="SATURATION", precision="double"
) as reader:
    saturation = reader.get_alldata()[:, :, 0, 0]
with flopy.utils.CellBudgetFile(
    workspace / f"{model_name}.cbc", precision="double"
) as reader:
    chd_flow = np.array(
        [rec["q"].sum() for rec in reader.get_data(text="CHD")]
    )  # positive = into the column
    surface_flow = np.array(
        [rec["q"].sum() for rec in reader.get_data(text="VSSB")]
    )  # positive = into the top soil cell

precipitation = obs["PRECIPITATION"].to_numpy() / area  # m/day
infiltration = (
    obs["INFILTRATION"].to_numpy() / area
)  # m/day, pond<->soil exchange, negative = exfiltration
runoff = -obs["RUNOFF"].to_numpy() / area  # m/day, leaving the model
pond_fill = (
    -obs["POND-STORAGE"].to_numpy() / area
)  # m/day, positive while the pond fills
pond_depth = obs["POND-DEPTH"].to_numpy()  # m
evaporation = (
    -obs["EVAPORATION"].to_numpy() / area
)  # m/day, total (pond + soil), positive leaving the model
phase_of_step = np.repeat(np.arange(len(schedule)), nsteps_phase)

# Split the total evaporation into its pond and soil parts.  The VSSB cell-budget record is the net
# surface exchange with the aquifer (infiltration minus the bare-soil evaporation taken from the top cell),
# so the bare-soil part is the difference with the pure infiltration observation and the pond part is the rest.
surface_exchange = surface_flow / area  # m/day, positive = net flow into the soil
soil_evaporation = (
    infiltration - surface_exchange
)  # m/day, bare-soil evaporation (>= 0)
pond_evaporation = (
    evaporation - soil_evaporation
)  # m/day, evaporation from the pond (>= 0)

# 1. budget closure per step; the pond balance uses the pond evaporation, while the bare-soil
#    evaporation is already contained in the surface exchange with the aquifer.
assert soil_evaporation.min() > -1e-9 and pond_evaporation.min() > -1e-9, (
    "negative evaporation split"
)
assert np.all(
    evaporation <= schedule["evaporation"].to_numpy()[phase_of_step] + 1e-9
), "evaporation above potential"
np.testing.assert_allclose(
    precipitation, infiltration + runoff + pond_fill + pond_evaporation, atol=1e-10
)
np.testing.assert_allclose(
    precipitation, schedule["precipitation"].to_numpy()[phase_of_step], atol=1e-12
)

# 2. pond consistency
np.testing.assert_allclose(np.cumsum(pond_fill * dt), pond_depth, atol=1e-9)
assert pond_depth.min() >= -1e-12 and pond_depth.max() <= max_pond_depth + 1e-9
assert np.all(pond_depth[runoff > 1e-9] >= max_pond_depth - 1e-6), (
    "runoff with a pond that is not full"
)

# 3. soil water balance
soil_water = (porosity * saturation * dz).sum(axis=1)  # m of water
initial_psi = initial_groundwater_head - z
m_vg = 1.0 - 1.0 / n_vg
initial_sat = residual_saturation + (1 - residual_saturation) * (
    1 + (alpha_vg * np.maximum(-initial_psi, 0.0)) ** n_vg
) ** (-m_vg)
initial_soil_water = (porosity * initial_sat * dz).sum()
inflow_to_soil = np.cumsum((surface_exchange + chd_flow / area) * dt)
np.testing.assert_allclose(soil_water - initial_soil_water, inflow_to_soil, atol=2e-5)

# 4. Darcy anchors: saturated column, full pond (no evaporation in these phases)
l_eff = -z[-1]
storm_phase = int(schedule.index[schedule["phase"].str.contains("storm")][0])
rise_phase = int(schedule.index[schedule["phase"].str.contains("rise")][0])
for phase_index in (storm_phase, rise_phase):
    last = np.flatnonzero(phase_of_step == phase_index)[-1]
    if pond_depth[last] < max_pond_depth - 1e-9:
        print(
            f"End of {schedule['phase'][phase_index]}: pond not full, Darcy check skipped"
        )
        continue
    expected = (
        ksat * (max_pond_depth - schedule["groundwater_head"][phase_index]) / l_eff
    )
    print(
        f"End of {schedule['phase'][phase_index]}: exchange {infiltration[last]:.5f} m/day, Darcy {expected:.5f} m/day"
    )
    np.testing.assert_allclose(infiltration[last], expected, rtol=1e-3)

print(
    f"Largest pond-budget residual: {np.abs(precipitation - infiltration - runoff - pond_fill - pond_evaporation).max():.1e} m/day"
)
print(
    f"Largest soil-water balance residual: {np.abs(soil_water - initial_soil_water - inflow_to_soil).max():.1e} m"
)

summary = (
    pd.DataFrame(
        {
            "precipitation": np.bincount(phase_of_step, precipitation * dt),
            "infiltration": np.bincount(
                phase_of_step, np.maximum(infiltration, 0) * dt
            ),
            "exfiltration": np.bincount(
                phase_of_step, np.maximum(-infiltration, 0) * dt
            ),
            "runoff": np.bincount(phase_of_step, runoff * dt),
            "pond evaporation": np.bincount(
                phase_of_step, np.maximum(pond_evaporation, 0) * dt
            ),
            "soil evaporation": np.bincount(
                phase_of_step, np.maximum(soil_evaporation, 0) * dt
            ),
            "pond depth at end": pond_depth[np.cumsum(nsteps_phase) - 1],
        },
        index=schedule["phase"],
    )
    * 1000.0
)
summary.columns = [
    f"{c} [mm]" if c != "pond depth at end" else "pond depth at end [mm]"
    for c in summary.columns
]
summary.round(1)

## Time series

Shaded bands alternate between the phases of the table above. From top to bottom:

* **Fluxes at the surface.** Precipitation (light blue), infiltration (blue, positive down) and exfiltration (orange, negative), and runoff (red line). In the storm, infiltration is limited by the soil, not by the rain: after a short, high initial capacity it settles at the Darcy value derived above, and the excess becomes runoff. In phase 5 runoff starts only when the pond is full.
* **Evaporation**, split into the part taken from the **pond** (cyan) and the part taken from the **soil** (olive), with the potential rate as a dashed line. While a pond exists (phase 3) the demand is met in full from the pond, so the total sits on the potential line. Once the pond is gone the demand moves to the soil, where it is cut by the relative permeability of the drying top cell and falls below the potential rate (phase 4).
* **Pond depth** with the maximum depth as a dashed line. The storm fills it in a few hours; in phase 3 it drains by infiltration and evaporation together; in phase 5 it fills by exfiltration alone, more slowly as the pond head approaches the soil head. Once it hits $d_\mathrm{max}$ the seepage becomes runoff.
* **Saturation** in the column; the black line is the water table ($\psi=0$). The rain raises it in phase 1, the storm saturates the top of the column, evaporation dries the top in phases 3-4, and in phase 5 it rises to the surface within hours because the capillary fringe above the water table is already almost saturated.
* **Groundwater head** at the bottom constant-head cell, the only groundwater forcing, with the land surface for reference.

In [ ]:
def step_series(values, initial=None):
    """Values per time step are rates over (t[k-1], t[k]]; use 'pre' steps."""
    first = values[0] if initial is None else initial
    return np.r_[0.0, times], np.r_[first, values]


def shade_phases(ax):
    for k, (start, end) in enumerate(zip(phase_start, phase_end)):
        if k % 2 == 0:
            ax.axvspan(start, end, color="0.92", zorder=0)


fig, axes = plt.subplots(
    5,
    1,
    figsize=(11, 14.5),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [2.2, 1.4, 1.4, 2.2, 1.2]},
)

ax = axes[0]
t, p = step_series(precipitation)
ax.fill_between(t, 0, p, step="pre", color="lightsteelblue", label="precipitation")
t, q = step_series(infiltration)
ax.fill_between(
    t, 0, q, where=q >= 0, step="pre", color="tab:blue", alpha=0.8, label="infiltration"
)
ax.fill_between(
    t,
    0,
    q,
    where=q < 0,
    step="pre",
    color="tab:orange",
    alpha=0.9,
    label="exfiltration",
)
t, r = step_series(runoff)
ax.step(t, r, where="pre", color="tab:red", linewidth=1.5, label="runoff")
ax.axhline(ksat, color="k", linestyle=":", linewidth=1, label="$K_\\mathrm{sat}$")
ax.set_ylabel("Flux at the surface [m/day]")
ax.legend(ncol=5, fontsize="small", loc="upper right")

ax = axes[1]
t, ep = step_series(pond_evaporation)
_, es = step_series(soil_evaporation)
ax.fill_between(
    t, 0, ep, step="pre", color="tab:cyan", alpha=0.9, label="from the pond"
)
ax.fill_between(
    t, ep, ep + es, step="pre", color="tab:olive", alpha=0.9, label="from the soil"
)
pot_times = np.r_[phase_start, phase_end[-1]]
ax.step(
    pot_times,
    np.r_[schedule["evaporation"], schedule["evaporation"].iloc[-1]],
    where="post",
    color="k",
    linestyle="--",
    linewidth=1,
    label="potential rate",
)
ax.set_ylabel("Evaporation [m/day]")
ax.legend(ncol=3, fontsize="small", loc="upper right")

ax = axes[2]
t, d = step_series(pond_depth, initial=0.0)
ax.step(t, d, where="pre", color="tab:blue", linewidth=2)
ax.axhline(
    max_pond_depth, color="k", linestyle="--", linewidth=1, label="max pond depth"
)
ax.set_ylabel("Pond depth [m]")
ax.legend(fontsize="small", loc="upper right")

ax = axes[3]
mesh = ax.pcolormesh(
    times,
    z,
    saturation.T,
    shading="nearest",
    cmap="Blues",
    vmin=np.floor(saturation.min() * 10) / 10,
    vmax=1.0,
)
ax.contour(times, z, pressure_head.T, levels=[0.0], colors="k", linewidths=1.5)
ax.plot([], [], "k-", label="water table ($\\psi=0$)")
ax.set_ylim(-column_length, 0.0)
ax.set_ylabel("Elevation z [m]")
ax.legend(fontsize="small", loc="lower right")
fig.colorbar(mesh, ax=ax, pad=0.01, label="Saturation [-]")

ax = axes[4]
gw_times = np.r_[phase_start, phase_end[-1]]
ax.step(
    gw_times,
    np.r_[schedule["groundwater_head"], schedule["groundwater_head"].iloc[-1]],
    where="post",
    color="tab:green",
    linewidth=2,
)
ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
ax.text(
    0.005,
    0.1,
    "land surface",
    transform=ax.get_yaxis_transform(),
    fontsize="small",
    va="bottom",
)
ax.set_ylabel("Groundwater head [m]")
ax.set_xlabel("Time [day]")

for ax in axes:
    shade_phases(ax)
    ax.grid(alpha=0.3)
    ax.set_xlim(0.0, phase_end[-1])
for k, (start, end, name) in enumerate(zip(phase_start, phase_end, schedule["phase"])):
    axes[0].text(
        (start + end) / 2,
        1.01,
        name.split()[0],
        transform=axes[0].get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontweight="bold",
    )
fig.suptitle(
    "VSSB: precipitation, infiltration, ponding, exfiltration, evaporation and runoff in one soil column"
)
plt.show()

## Water balance and profiles

Left: cumulative water per unit area. The full-system balance closes at every step: cumulative precipitation equals the net infiltration into the soil (infiltration minus exfiltration and bare-soil evaporation) plus runoff, total evaporation and the water held in the pond. In phases 3-4 there is no rain, so evaporation and net infiltration are the only sinks and the net infiltration becomes negative while the top of the soil loses water; in phase 5 the pond and the runoff are fed by groundwater. Middle and right: saturation and pressure head profiles at the **end of every phase**, with the initial state dashed. Positive pressure head means saturated soil below the water table.

In [ ]:
cum_precipitation = np.cumsum(precipitation * dt)
cum_infiltration = np.cumsum(
    surface_exchange * dt
)  # net water delivered to the soil (incl. bare-soil evaporation)
cum_runoff = np.cumsum(runoff * dt)
cum_evaporation = np.cumsum(evaporation * dt)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
ax = axes[0]
for series, label, style in [
    (cum_precipitation, "precipitation", {"color": "lightsteelblue", "linewidth": 5}),
    (cum_infiltration, "net infiltration", {"color": "tab:blue"}),
    (cum_runoff, "runoff", {"color": "tab:red"}),
    (cum_evaporation, "evaporation", {"color": "tab:olive"}),
    (pond_depth, "pond", {"color": "tab:purple"}),
]:
    ax.plot(times, series * 1000.0, label=label, **style)
ax.plot(
    times,
    (cum_infiltration + cum_runoff + cum_evaporation + pond_depth) * 1000.0,
    "k--",
    linewidth=1,
    label="infiltration + runoff + evaporation + pond",
)
ax.set_xlabel("Time [day]")
ax.set_ylabel("Cumulative water [mm]")
ax.legend(fontsize="small")
ax.grid(alpha=0.3)

end_index = np.cumsum(nsteps_phase) - 1
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(schedule)))
for ax, values, initial, label in [
    (axes[1], saturation, initial_sat, "Saturation [-]"),
    (axes[2], pressure_head, initial_psi, "Pressure head $\\psi$ [m]"),
]:
    ax.plot(initial, z, "k--", label="initial")
    for k, (index, color) in enumerate(zip(end_index, colors)):
        ax.plot(
            values[index],
            z,
            color=color,
            label=f"end of {schedule['phase'][k]} ({times[index]:g} d)",
        )
    ax.set_xlabel(label)
    ax.set_ylim(-column_length, 0.0)
    ax.grid(alpha=0.3)
axes[1].set_ylabel("Elevation z [m]")
axes[1].legend(fontsize="x-small", loc="lower left")
plt.show()

## What to look at, and things to try

* **Capacity versus rain.** In phase 1 infiltration equals the rain. In the storm it is set by the soil and the pond head, not by the rain, so $P-Q_\mathrm{inf}$ first fills the pond and then runs off. With a saturated column it equals $K_\mathrm{sat}(d_\mathrm{max}-h_\mathrm{bottom})/L_\mathrm{eff}$, which is below $K_\mathrm{sat}$ (dotted) because the head difference is smaller than the column length.
* **Evaporation: pond versus soil.** In phase 3 a pond is present, so the whole potential demand is met from the pond and the evaporation panel sits on the dashed potential line. As soon as the pond is gone the demand passes to the soil (phase 4), where it is multiplied by the relative permeability $k_r$ of the top cell; as that cell dries, $k_r$ drops and the evaporation becomes supply-limited and falls well below the potential rate. The pond evaporation is part of the pond budget, while the bare-soil evaporation is pulled straight from the soil and shows up in the surface exchange, which is why the soil-water balance uses the `VSSB` cell-budget record rather than the infiltration observation.
* **Pond versus groundwater.** In phase 5 exfiltration is largest when the pond is empty ($K_\mathrm{sat}\,h_\mathrm{bottom}/L_\mathrm{eff}$) and decreases as the pond deepens, since $Q=C_s(z_s+d-h)$. With `max_pond_depth = 0.2` the pond never fills by seepage alone: it approaches the groundwater head (0.15 m) asymptotically, exfiltration fades to zero and there is no seepage runoff.
* **Rain on a seeping surface** (phase 6): precipitation does not reduce exfiltration, because the pond is already full; the runoff is rain plus seepage.
* Raise the potential evaporation of phase 4 (try 0.3 m/day) to see that the extra demand barely increases the actual soil evaporation once $k_r$ is small, or lengthen phase 3 to watch the pond disappear entirely by infiltration and evaporation.
* Set `max_pond_depth = 0` to see that all water that cannot infiltrate immediately leaves as runoff, without storage, and that evaporation then has no pond to draw from.
* Make the rain in phase 1 larger than the capacity (try 0.3 m/day) for infiltration-excess runoff on the initially drier soil, or lower the groundwater head of phase 5 below 0 to see that exfiltration then never starts.
* **Resolution.** The surface conductance uses the gradient over half the top cell, $\Delta z/2$, a first-node approximation that can underestimate the very high capacity of a dry soil at the start of rain, and likewise controls how fast the top cell dries under bare-soil evaporation. For this wet profile it hardly matters: with 25, 50 or 200 layers the maximum infiltration in the storm differs by less than 1 %.

## Animation: saturation and the pond through time

Finally, a small animation of the whole run. The soil column is coloured by saturation, with the water table (where $\psi=0$) as a line, and the pond sits in a box on top of the land surface, its water level rising and falling with `pond_depth`. The frames are subsampled to keep the embedded animation light; use the slider or the play button to scrub through the phases.

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle

plt.rcParams["animation.embed_limit"] = 50  # MB, for the base64-embedded frames

width = np.sqrt(area)  # m, column width for the drawing
phase_name = schedule["phase"].to_numpy()[phase_of_step]

# water-table elevation per step: shallowest depth where the pressure head crosses zero
water_table = np.minimum(0.0, np.full(len(times), botm[-1]))
for k in range(len(times)):
    saturated = pressure_head[k] >= 0.0
    if saturated.any():
        i = int(np.argmax(saturated))
        if i == 0:
            water_table[k] = 0.0
        else:
            p0, p1 = pressure_head[k, i - 1], pressure_head[k, i]
            water_table[k] = min(
                0.0, z[i - 1] + (0.0 - p0) / (p1 - p0) * (z[i] - z[i - 1])
            )

# subsample the time steps so the embedded animation stays small
stride = max(1, len(times) // 180)
frames = np.unique(np.r_[np.arange(0, len(times), stride), len(times) - 1])

fig, ax = plt.subplots(figsize=(3.6, 6.4), constrained_layout=True)
x_edges = np.array([0.0, width])
z_edges = np.r_[0.0, botm]
mesh = ax.pcolormesh(
    x_edges,
    z_edges,
    saturation[0][:, None],
    shading="flat",
    cmap="Blues",
    vmin=np.floor(saturation.min() * 10) / 10,
    vmax=1.0,
)
fig.colorbar(mesh, ax=ax, pad=0.02, shrink=0.6, label="Saturation [-]")

# the pond "tank" on top of the land surface: an outline and a filling water level
ax.add_patch(
    Rectangle(
        (0.0, 0.0), width, max_pond_depth, fill=False, edgecolor="0.4", linewidth=1.2
    )
)
pond_water = ax.add_patch(
    Rectangle(
        (0.0, 0.0),
        width,
        pond_depth[0],
        facecolor="tab:blue",
        alpha=0.85,
        edgecolor="none",
    )
)
ax.axhline(0.0, color="k", linewidth=1.5)  # land surface
(wt_line,) = ax.plot(
    [0.0, width], [water_table[0]] * 2, color="k", linewidth=1.2, linestyle="--"
)
ax.plot([], [], "k--", linewidth=1.2, label="water table ($\\psi=0$)")

ax.set_xlim(0.0, width)
ax.set_ylim(botm[-1], max_pond_depth * 1.25)
ax.set_xticks([])
ax.set_ylabel("Elevation z [m]")
ax.legend(loc="lower left", fontsize="small")
title = ax.set_title("")


def update(k):
    mesh.set_array(saturation[k])
    pond_water.set_height(pond_depth[k])
    wt_line.set_ydata([water_table[k]] * 2)
    title.set_text(
        f"{phase_name[k]}\n$t = {times[k]:.2f}$ d,  pond $= {pond_depth[k] * 1000:.1f}$ mm"
    )
    return mesh, pond_water, wt_line, title


anim = FuncAnimation(fig, update, frames=frames, interval=120, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())